# CallWhisper-8k: Whisper-small LoRA Edge Smoke Test

Purpose: start the compact-model adaptation track.

This notebook is intentionally a smoke test first. It proves that the training pipeline works and produces an adapter. It should not be reported as a final model result unless the train data is a real training split and excludes frozen benchmark files.

Serious run target: `GV_Train_100h` or another separate training split.

Frozen held-out eval manifests:

- `datasets/manifests/gramvaani_dev_50.csv`
- `datasets/manifests/gramvaani_dev_50_8khz.csv`
- `datasets/manifests/gramvaani_dev_50_highrate.csv`
- FLEURS clean-control manifest in Drive

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/anshulLuhsna/CallWhisper-8k.git"
REPO_DIR = Path("/content/CallWhisper-8k")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/call-whisper")
CHECKPOINT_DIR = DRIVE_PROJECT_DIR / "checkpoints/whisper-small-lora-gramvaani-smoke"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo:", Path.cwd())
print("Drive project dir:", DRIVE_PROJECT_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -m pip install -U pip
!python -m pip install -e ".[finetune]"
!python -m pip install -q soundfile librosa jiwer
!apt-get update -qq && apt-get install -y -qq ffmpeg


## Link Drive Data

For a smoke test, this notebook can use `GV_Dev_5h` minus frozen benchmark IDs. For a serious training run, upload `GV_Train_100h` to Drive and set `TRAIN_DATASET_DIR` to that folder.

In [ ]:
def link_or_keep(source: Path, target: Path) -> None:
    if target.exists() or target.is_symlink():
        return
    if not source.exists():
        raise FileNotFoundError(f"Missing source: {source}")
    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)

link_or_keep(DRIVE_PROJECT_DIR / "GV_Dev_5h", REPO_DIR / "datasets/GV_Dev_5h")
if (DRIVE_PROJECT_DIR / "GV_Train_100h").exists():
    link_or_keep(DRIVE_PROJECT_DIR / "GV_Train_100h", REPO_DIR / "datasets/GV_Train_100h")

# Change this to REPO_DIR / "datasets/GV_Train_100h" for a serious run.
TRAIN_DATASET_DIR = REPO_DIR / "datasets/GV_Dev_5h"
SMOKE_TEST_ONLY = TRAIN_DATASET_DIR.name == "GV_Dev_5h"

print("Train dataset dir:", TRAIN_DATASET_DIR)
print("Smoke test only:", SMOKE_TEST_ONLY)


## Build Train/Eval Rows

The frozen benchmark IDs are excluded from training. This is mandatory.

In [ ]:
import csv
from pathlib import Path

def read_text(path: Path) -> dict[str, str]:
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            utt, _, text = line.partition(" ")
            if utt and text:
                rows[utt] = text.strip()
    return rows

def read_scp(path: Path) -> dict[str, Path]:
    rows = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                rows[parts[0]] = TRAIN_DATASET_DIR / parts[1]
    return rows

def frozen_ids_from_manifest(path: Path) -> set[str]:
    ids = set()
    with path.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            ids.add(Path(row["audio_path"]).stem)
    return ids

frozen_ids = set()
for manifest in [
    REPO_DIR / "datasets/manifests/gramvaani_dev_50.csv",
    REPO_DIR / "datasets/manifests/gramvaani_dev_50_8khz.csv",
    REPO_DIR / "datasets/manifests/gramvaani_dev_50_highrate.csv",
]:
    frozen_ids |= frozen_ids_from_manifest(manifest)

texts = read_text(TRAIN_DATASET_DIR / "text")
audio_paths = read_scp(TRAIN_DATASET_DIR / "mp3.scp")

rows = []
for utt, transcript in texts.items():
    if utt in frozen_ids:
        continue
    audio_path = audio_paths.get(utt)
    if not audio_path or not audio_path.exists():
        continue
    if "<incomplete>" in transcript:
        continue
    rows.append({"audio": str(audio_path), "text": transcript, "utt_id": utt})

SMOKE_TRAIN_LIMIT = 200
SMOKE_EVAL_LIMIT = 30
train_rows = rows[:SMOKE_TRAIN_LIMIT]
eval_rows = rows[SMOKE_TRAIN_LIMIT:SMOKE_TRAIN_LIMIT + SMOKE_EVAL_LIMIT]

print("Frozen IDs excluded:", len(frozen_ids))
print("Available non-frozen rows:", len(rows))
print("Train rows:", len(train_rows))
print("Eval rows:", len(eval_rows))
assert train_rows and eval_rows


## Prepare Whisper-small + LoRA

In [ ]:
import torch
from datasets import Audio, Dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

BASE_MODEL = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="Hindi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = Dataset.from_list(train_rows).cast_column("audio", Audio(sampling_rate=16000))
eval_ds = Dataset.from_list(eval_rows).cast_column("audio", Audio(sampling_rate=16000))


In [ ]:
def prepare_example(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_ds = train_ds.map(prepare_example, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(prepare_example, remove_columns=eval_ds.column_names)
print(train_ds)
print(eval_ds)


## Train Smoke Adapter

This is intentionally tiny. Increase steps only after the smoke run completes and eval works.

In [ ]:
from dataclasses import dataclass
from typing import Any
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=10,
    max_steps=100,
    fp16=torch.cuda.is_available(),
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    predict_with_generate=False,
    report_to=["tensorboard"],
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    tokenizer=processor.feature_extractor,
)

trainer.train()
trainer.save_model(str(CHECKPOINT_DIR / "final_adapter"))
processor.save_pretrained(str(CHECKPOINT_DIR / "processor"))
print("Saved adapter:", CHECKPOINT_DIR / "final_adapter")


## Next Evaluation Step

After the smoke adapter trains, add a dedicated evaluation cell that loads the base model + adapter and writes benchmark JSON for:

- `gramvaani_dev_50`
- `gramvaani_dev_50_8khz`
- `gramvaani_dev_50_highrate`
- `fleurs_hi_clean_50`

Do not claim anything from training loss alone.